# 04. 学習済モデルで自動プレイ & 提出用ファイルの作成

`03_train_bc.ipynb` でダウンロードした `.pt` ファイルを `../models/checkpoints/` に配置してから実行してください。

`STACK_SIZE` はチェックポイントから読み取るので、`03` で 1〜4 のどの値で学習した場合でも
`CHECKPOINT_PATH` を差し替えるだけで動きます。このノートブック側で合わせる必要はありません。

AIのプレイ (1エピソード) と提出物の作成は同じ実行で行います。動画・CSV・zip がまとめて
`results/submissions/` に出力されます。

In [ ]:
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

ENV_ID = "SuperMarioBros-1-1-v0"

env = JoypadSpace(gym_super_mario_bros.make(ENV_ID, render_mode="human"), SIMPLE_MOVEMENT)

# Same decimation as 01_collect_data.ipynb. The training data updates its frame
# history at 30fps (once every 2 frames), so inference picks an action at the
# same rate and holds it in between (frame skip / action repeat).
TARGET_FPS = 30
RECORD_EVERY_N = round(env.metadata["render_fps"] / TARGET_FPS)

print("SIMPLE_MOVEMENT:", SIMPLE_MOVEMENT)
print("make sure this ordering matches the one used in 03_train_bc.ipynb")
print(f"render_fps={env.metadata['render_fps']} -> inference fps={TARGET_FPS} (RECORD_EVERY_N={RECORD_EVERY_N})")

In [ ]:
# --- Keep this block identical in 03_train_bc.ipynb and 04_playagent.ipynb ---
from collections import deque

import cv2
import numpy as np
import torch
import torch.nn as nn

FRAME_SIZE = (84, 84)

# --- Exercise knob: how many consecutive frames the model sees at once -----
# 1 turns this into an ordinary single-image classifier; 4 is the usual choice
# in the Atari literature. Try values in the 1-4 range and compare how well the
# agent actually plays.
#
# A single frame cannot show velocity. Whether Mario is rising or falling in a
# jump, and how fast he is moving, only becomes visible across several frames.
# Stacking more frames makes that observable, at the cost of a larger input and
# more memory (the dataset holds STACK_SIZE copies of every frame).
#
# Set this in 03 only. 04_playagent.ipynb reads the value back out of the
# checkpoint, so there is nothing to keep in sync there by hand.
STACK_SIZE = 4
# ---------------------------------------------------------------------------

NUM_ACTIONS = len(SIMPLE_MOVEMENT)


def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    """Convert an RGB frame to grayscale and resize it to 84x84."""
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    return cv2.resize(gray, FRAME_SIZE, interpolation=cv2.INTER_AREA)


class FrameStacker:
    """Stack the most recent STACK_SIZE preprocessed frames."""

    def __init__(self, stack_size: int = STACK_SIZE):
        self.stack_size = stack_size
        self.frames = deque(maxlen=stack_size)

    def reset(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.clear()
        for _ in range(self.stack_size):
            self.frames.append(processed_frame)
        return self.get()

    def step(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.append(processed_frame)
        return self.get()

    def get(self) -> np.ndarray:
        return np.stack(self.frames, axis=0)


class MarioCNN(nn.Module):
    def __init__(self, num_actions: int = NUM_ACTIONS, in_channels: int = STACK_SIZE):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, *FRAME_SIZE)
            n_flatten = self.features(dummy).shape[1]
        self.head = nn.Sequential(
            nn.Linear(n_flatten, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.float() / 255.0
        x = self.features(x)
        return self.head(x)
# --- end of shared block ---

In [ ]:
from pathlib import Path

# change this to the file you downloaded from 03_train_bc.ipynb
CHECKPOINT_PATH = Path("../models/checkpoints/mario_bc_s4_YYYYMMDD_HHMMSS.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

# 03 stores the STACK_SIZE it trained with, so the model is always rebuilt to
# match the checkpoint. Without this, training with STACK_SIZE=1 and playing
# with the default 4 would fail here with an opaque tensor-shape error.
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
    STACK_SIZE = checkpoint["stack_size"]
else:
    # A bare state_dict, saved before checkpoints carried metadata. The first
    # conv layer's weight has shape (out_channels, in_channels, 8, 8), so the
    # stack size it was trained with can be read straight off the weights.
    state_dict = checkpoint
    STACK_SIZE = state_dict["features.0.weight"].shape[1]
    print(f"no metadata in this checkpoint - inferred STACK_SIZE={STACK_SIZE} from its weights")

# Pass STACK_SIZE explicitly. The default arguments of MarioCNN and FrameStacker
# were bound when the classes were defined, so reassigning the global just above
# does not reach them.
model = MarioCNN(in_channels=STACK_SIZE).to(device)
model.load_state_dict(state_dict)
model.eval()
print(f"loaded: {CHECKPOINT_PATH} (device={device}, STACK_SIZE={STACK_SIZE})")

In [ ]:
import time
from datetime import datetime

# 自分の学籍番号に書き換えること。提出ファイル名になる。
STUDENT_ID = "s00000"

# AIのプレイと提出物の作成は同じ実行で行う。1回の実行 = 1回の挑戦 = 提出物1式。
# 何度か挑戦して良いものを提出したい場合は、このセルを実行し直す(毎回別のフォルダになる)。
EPISODES = 1

RESULTS_DIR = Path("../results")
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
submission_dir = RESULTS_DIR / "submissions" / f"{STUDENT_ID}_{RUN_TS}"
submission_dir.mkdir(parents=True, exist_ok=True)

ACTION_NAMES = ["_".join(a) if a else "NOOP" for a in SIMPLE_MOVEMENT]

# One row per emulator frame. This is the per-frame log that goes into the
# submission: it lines up 1:1 with the frames of the mp4, so the video and the
# numbers can always be checked against each other.
frame_rows = []

fps = env.metadata.get("render_fps", 60)
frame_duration = 1 / fps

for episode in range(EPISODES):
    obs, info = env.reset()
    stacker = FrameStacker(STACK_SIZE)
    stack = stacker.reset(preprocess_frame(obs))

    # written straight into the submission folder, so there is no second copy
    video_path = submission_dir / f"ep{episode:02d}.mp4"
    h, w = obs.shape[0], obs.shape[1]
    video_writer = cv2.VideoWriter(
        str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
    )

    steps = 0
    frame_idx = 0
    terminated = truncated = False
    while not (terminated or truncated):
        stack_tensor = torch.from_numpy(stack).unsqueeze(0).to(device)
        with torch.no_grad():
            action = model(stack_tensor).argmax(dim=1).item()

        # Match the training data's decimation (one action every
        # RECORD_EVERY_N frames) by holding the action for that many frames
        for _ in range(RECORD_EVERY_N):
            t0 = time.time()

            obs, reward, terminated, truncated, info = env.step(action)
            steps += 1

            env.render()
            video_writer.write(cv2.cvtColor(obs, cv2.COLOR_RGB2BGR))

            # info["time"] counts down from 400, so it is recorded as
            # time_remaining to match the column name used in 01/02.
            frame_rows.append({
                "episode": episode,
                "frame_idx": frame_idx,
                "action": action,
                "action_name": ACTION_NAMES[action],
                "x_pos": info["x_pos"],
                "y_pos": info["y_pos"],
                "x_pos_max": info["x_pos_max"],
                "coins": info["coins"],
                "score": info["score"],
                "time_remaining": info["time"],
                "flag_get": info["flag_get"],
                "status": info["status"],
                "life": info["life"],
            })
            frame_idx += 1

            elapsed = time.time() - t0
            if elapsed < frame_duration:
                time.sleep(frame_duration - elapsed)

            if terminated or truncated:
                break

        stack = stacker.step(preprocess_frame(obs))

    video_writer.release()

    print(
        f"episode {episode}: score={info['score']} x_pos_max={info['x_pos_max']} "
        f"coins={info['coins']} flag_get={info['flag_get']} steps={steps} "
        f"video={video_path.name}"
    )

env.close()

## 提出用ファイルの作成

プレイ中に動画は既に `results/submissions/<STUDENT_ID>_<timestamp>/` へ直接書き込まれている。
このセルはそこに2つのCSVを追加して、提出用のzipにまとめる。

| ファイル | 内容 |
|---|---|
| `epNN.mp4` | AIのプレイ動画 |
| `frames.csv` | **フレームごと**のログ。マリオの座標 (`x_pos`, `y_pos`)、選択したアクション、コイン数、スコア、残り時間、ゴール到達 (`flag_get`) など |
| `episodes.csv` | エピソードごとの集計と **Skillスコア** |

`frames.csv` の行はmp4のフレームと1対1で対応しているので、動画と数値を突き合わせて確認できる。
Skillスコアは `episodes.csv` に入っているが、`frames.csv` だけからでも再計算できる。

In [ ]:
import zipfile

import pandas as pd

# --- Skill score: keep these constants and the formula identical to 01/02 ---
X_GOAL = 3161  # x position of the flagpole in world 1-1
T_FAST = 110  # full speed score at or below this time
T_SLOW = 260  # zero speed score at or above this time
COIN_TARGET = 10  # full collection score at this many coins

WEIGHTS = {"progress": 0.45, "clear": 0.30, "speed": 0.15, "collect": 0.10}


def add_skill_score(df: pd.DataFrame) -> pd.DataFrame:
    """Return the manifest with a Skill score and its component terms added."""
    out = df.copy()
    out["progress"] = (out["x_pos_max"] / X_GOAL).clip(upper=1.0)
    out["clear"] = out["flag_get"].astype(int)
    out["t_used"] = 400 - out["time_remaining"]
    out["speed"] = out["clear"] * ((T_SLOW - out["t_used"]) / (T_SLOW - T_FAST)).clip(0, 1)
    out["collect"] = (out["coins"] / COIN_TARGET).clip(upper=1.0)
    out["skill"] = 100 * sum(WEIGHTS[k] * out[k] for k in WEIGHTS)
    return out
# --- end of shared block ---


# 1. the per-frame log
frames_df = pd.DataFrame(frame_rows)
frames_df.to_csv(submission_dir / "frames.csv", index=False, encoding="utf-8")

# 2. per-episode summary, derived from the per-frame log so the two can never
#    disagree. x_pos_max/coins/score only grow and time_remaining only falls, so
#    max/min over the episode is the value at the end of it.
per_episode = (
    frames_df.groupby("episode")
    .agg(
        num_frames=("frame_idx", "count"),
        x_pos_max=("x_pos_max", "max"),
        coins=("coins", "max"),
        score=("score", "max"),
        time_remaining=("time_remaining", "min"),
        flag_get=("flag_get", "max"),
    )
    .reset_index()
)
episodes_df = add_skill_score(per_episode)
episodes_df["checkpoint"] = CHECKPOINT_PATH.name
episodes_df["stack_size"] = STACK_SIZE
episodes_df.to_csv(submission_dir / "episodes.csv", index=False, encoding="utf-8")

# 3. one zip to hand in (the videos are already sitting in submission_dir)
videos = sorted(submission_dir.glob("*.mp4"))
zip_path = submission_dir.parent / f"submission_{STUDENT_ID}_{RUN_TS}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(submission_dir.rglob("*")):
        zf.write(path, path.relative_to(submission_dir.parent))

print(f"=== submission for {STUDENT_ID} ===")
print(f"frames.csv : {len(frames_df)} rows x {len(frames_df.columns)} columns")
print(f"videos     : {len(videos)}")
print(f"distance   : {episodes_df['x_pos_max'].mean():.0f} / {X_GOAL}")
print(f"cleared    : {'yes' if episodes_df['clear'].any() else 'no'}")
print(f"skill      : {episodes_df['skill'].mean():.1f}")
print(f"\nsubmit this file: {zip_path}  ({zip_path.stat().st_size / 1024**2:.1f} MB)")

episodes_df[["episode", "num_frames", "x_pos_max", "coins", "time_remaining",
             "flag_get", "progress", "speed", "collect", "skill"]].round(2)